System Path Setup

In [1]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [2]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import pandas as pd
import geopandas as gpd
import folium # For interactive mapping
from configs.regions import kenyan_coast_roi # Your ROI
from configs.carbon_coefficients import (
    CARBON_FRACTION_BIOMASS,
    BGB_AGB_RATIO,
    SOIL_CARBON_DENSITY_PER_M,
    DEFAULT_SOIL_DEPTH_M
) # For context/future calc

ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID
print("All core libraries imported and GEE initialized.")

Region of Interest for Kenyan Coast defined.
Carbon coefficients loaded.
All core libraries imported and GEE initialized.


 Load Consolidated Carbon Analysis Image and Calculate Total AGC Stock

In [3]:
# Cell 3: Load Consolidated Carbon Analysis Image and Calculate Total AGC Stock
print("--- Calculating Total Aboveground Carbon (AGC) Stock ---")

# Load the consolidated carbon analysis image from GEE Assets (exported from Task 3.3)
my_gee_project_id = 'gaias-ark' # <--- CONFIRM YOUR GEE PROJECT ID
consolidated_carbon_asset_id = f'projects/{my_gee_project_id}/assets/gaias_ark_carbon_analysis_image_kenya'

try:
    carbon_analysis_image = ee.Image(consolidated_carbon_asset_id)
    _ = carbon_analysis_image.getInfo() # Force evaluation to check existence
    print(f"Loaded consolidated carbon image asset: {consolidated_carbon_asset_id}")
except Exception as e:
    print(f"ERROR: Consolidated carbon image asset not found or inaccessible: {e}")
    print("Please ensure the GEE export task for this asset completed successfully.")
    sys.exit("Cannot proceed with carbon analysis.")

# Access the AGC_Density_tonnes_C_ha band
agc_density_image = carbon_analysis_image.select('AGC_Density_tonnes_C_ha')

# Calculate pixel area in hectares
pixel_area_ha = ee.Image.pixelArea().divide(10000)

# Calculate total AGC in tonnes of Carbon by summing (AGC_Density * pixel_area_ha) over the ROI
total_agc_tonnes = agc_density_image.multiply(pixel_area_ha) \
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=kenyan_coast_roi,
        scale=30, # Use same scale as image
        maxPixels=1e10 # For large computations
    ).get('AGC_Density_tonnes_C_ha') # Get the sum for the 'AGC_Density_tonnes_C_ha' band

# Fetch the result from GEE
total_agc_tonnes_value = total_agc_tonnes.getInfo()

print(f"\nCalculated Total Aboveground Carbon (AGC) Stock for Kenyan Coastal ROI: {total_agc_tonnes_value:,.2f} tonnes C")

# Optional: Calculate total mangrove area in hectares for context (from 'mangrove_presence' band)
mangrove_area_image = carbon_analysis_image.select('mangrove_presence').multiply(pixel_area_ha)
total_mangrove_area_ha = mangrove_area_image.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=kenyan_coast_roi,
    scale=30,
    maxPixels=1e10
).get('mangrove_presence').getInfo()

print(f"Total Mangrove Area in ROI: {total_mangrove_area_ha:,.2f} hectares")

# Simple Sanity Check
if total_mangrove_area_ha > 0:
    avg_agc_per_ha = total_agc_tonnes_value / total_mangrove_area_ha
    print(f"Average AGC per hectare: {avg_agc_per_ha:,.2f} tonnes C/ha")
    if avg_agc_per_ha < 1 or avg_agc_per_ha > 300: # Very rough sanity check
        print("WARNING: Average AGC per hectare seems outside typical mangrove ranges (1-300 tC/ha).")

--- Calculating Total Aboveground Carbon (AGC) Stock ---
Loaded consolidated carbon image asset: projects/gaias-ark/assets/gaias_ark_carbon_analysis_image_kenya

Calculated Total Aboveground Carbon (AGC) Stock for Kenyan Coastal ROI: 25,444.20 tonnes C
Total Mangrove Area in ROI: 31,676.78 hectares
Average AGC per hectare: 0.80 tonnes C/ha


Visualize AGC Density Distribution

In [9]:
# Cell 4: Visualize AGC Density Distribution (REVISED: Debugging File Save)
print("--- Visualizing Aboveground Carbon (AGC) Density ---")

agc_density_image_viz = carbon_analysis_image.select('AGC_Density_tonnes_C_ha')

agc_vis_params = {
    'min': 0, 'max': 10,
    'palette': ['#ffffcc', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8', '#0c2c84']
}

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

print("Attempting to create Folium map object...")
m_agc_map = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')
print("Folium map object created.")

folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#00000000', 'color': 'red', 'weight': 3, 'fillOpacity': 0.0}
).add_to(m_agc_map)

map_id_dict_agc_viz = agc_density_image_viz.getMapId(agc_vis_params)
folium.TileLayer(
    tiles=map_id_dict_agc_viz['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='AGC Density (tonnes C/ha)'
).add_to(m_agc_map)

mangrove_presence_image_viz = carbon_analysis_image.select('mangrove_presence')
mangrove_vis_params = {
    'min': 0, 'max': 1,
    'palette': ['#00000000', 'green']
}
map_id_dict_presence = mangrove_presence_image_viz.getMapId(mangrove_vis_params)
folium.TileLayer(
    tiles=map_id_dict_presence['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Mangrove Presence (GMW)',
    show=True
).add_to(m_agc_map)

folium.LayerControl().add_to(m_agc_map)

# --- Debugging file path and save ---
output_map_path = os.path.join(project_root, 'docs', 'agc_density_map.html')
print(f"Attempting to save map to: {output_map_path}")

# Add a try-except block to catch potential file write errors
try:
    m_agc_map.save(output_map_path)
    print(f"Interactive map successfully saved to: {output_map_path}")
    print("Open this HTML file in your web browser to view the map.")
except Exception as e:
    print(f"ERROR: Failed to save map to HTML file: {e}")
    print("Please check file path, permissions, or disk space.")

--- Visualizing Aboveground Carbon (AGC) Density ---
Attempting to create Folium map object...
Folium map object created.
Attempting to save map to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\docs\agc_density_map.html
Interactive map successfully saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\docs\agc_density_map.html
Open this HTML file in your web browser to view the map.


Statistical Analysis of AGC Density

In [10]:
# Cell 5: Statistical Analysis of AGC Density
print("--- Performing Statistical Analysis on AGC Density ---")

# Load the consolidated carbon analysis image (if not already loaded in session)
my_gee_project_id = 'gaias-ark'
consolidated_carbon_asset_id = f'projects/{my_gee_project_id}/assets/gaias_ark_carbon_analysis_image_kenya'

try:
    carbon_analysis_image = ee.Image(consolidated_carbon_asset_id)
    _ = carbon_analysis_image.getInfo() # Force evaluation
except Exception as e:
    print(f"ERROR: Consolidated carbon image asset not found: {e}")
    print("Please ensure the GEE export task for this asset completed successfully.")
    sys.exit("Cannot proceed with carbon analysis.")

# Access the AGC_Density_tonnes_C_ha and mangrove_presence bands
agc_density_image = carbon_analysis_image.select('AGC_Density_tonnes_C_ha')
mangrove_presence_image = carbon_analysis_image.select('mangrove_presence')

# --- 1. Basic statistics of AGC Density within the ROI ---
# We want to know the min, max, mean, standard deviation of AGC density.
# Reduce the AGC density image over the ROI, but *only where mangroves are present*.
# Use the mangrove_presence_image as a mask for this reduction.
agc_stats = agc_density_image.updateMask(mangrove_presence_image).reduceRegion(
    reducer=ee.Reducer.min()
            .combine(ee.Reducer.max(), None, True)
            .combine(ee.Reducer.mean(), None, True)
            .combine(ee.Reducer.stdDev(), None, True)
            .combine(ee.Reducer.median(), None, True),
    geometry=kenyan_coast_roi,
    scale=30,
    maxPixels=1e10
)
print("\nAGC Density Statistics (tonnes C/ha, only within mangroves):")
for stat, value in agc_stats.getInfo().items():
    # The keys will be like 'AGC_Density_tonnes_C_ha_min'
    print(f"  {stat.replace('AGC_Density_tonnes_C_ha_', '')}: {value:,.2f}")


# --- 2. Relationship between Mangrove Presence and AGC Density ---
# We already have `mangroves_filtered` implicitly in `carbon_analysis_image`
# and know average AGC per hectare (0.80 tC/ha).

# Let's get pixel counts for mangrove presence (for histogram-like insight)
# This might be redundant with total area calc, but good for direct pixel count.
# We create a categorical image (0=non-mangrove, 1=mangrove) and count pixels.
mangrove_pixel_counts = mangrove_presence_image.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=kenyan_coast_roi,
    scale=30,
    maxPixels=1e10
).getInfo()

print("\nMangrove Presence Pixel Counts:")
# The output of frequencyHistogram is a dictionary where keys are values and values are counts.
# It typically returns a dictionary like {'0': count_of_zeros, '1': count_of_ones}
if 'mangrove_presence' in mangrove_pixel_counts:
    presence_counts = mangrove_pixel_counts['mangrove_presence']
    non_mangrove_pixels = presence_counts.get('0', 0)
    mangrove_pixels = presence_counts.get('1', 0)
    print(f"  Non-mangrove pixels in ROI: {non_mangrove_pixels:,.0f}")
    print(f"  Mangrove pixels in ROI: {mangrove_pixels:,.0f}")
else:
    print("  Could not retrieve mangrove pixel counts (possible empty image or no data).")

# --- 3. (Advanced, Optional for MVP) Investigate areas of high/low AGC Density ---
# This would involve querying areas based on AGC thresholds. For MVP, the stats are sufficient.

--- Performing Statistical Analysis on AGC Density ---

AGC Density Statistics (tonnes C/ha, only within mangroves):
  max: 1,032.78
  mean: 0.78
  median: 0.17
  min: 0.00
  stdDev: 7.67

Mangrove Presence Pixel Counts:
  Non-mangrove pixels in ROI: 12,298,284
  Mangrove pixels in ROI: 354,771


Visualize High AGC Density Areas

In [13]:
# Cell 6 (Optional): Visualize High AGC Density Areas (REVISED VISUALIZATION - Mangrove Visibility)
print("--- Visualizing Areas of High Aboveground Carbon (AGC) Density (Revised) ---")

agc_density_image = carbon_analysis_image.select('AGC_Density_tonnes_C_ha')
mangrove_presence_image = carbon_analysis_image.select('mangrove_presence')

high_carbon_threshold = 20
extreme_carbon_threshold = 100

agc_density_masked = agc_density_image.updateMask(mangrove_presence_image.eq(1))

agc_vis_params_revised = {
    'min': 0, 'max': 150,
    'palette': ['#f7fcf0', '#e0f3db', '#ccebc5', '#a8ddb5', '#7bccc4', '#4eb3d3', '#2b8cbe', '#0868ac', '#084081']
}

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

m_revised_agc = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='CartoDB positron')

# Add ROI (transparent fill)
folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#00000000', 'color': 'red', 'weight': 3, 'fillOpacity': 0.0}
).add_to(m_revised_agc)

# --- CRITICAL CHANGE: Add Mangrove Presence layer FIRST, and make it opaque green ---
# We want this to be the base visible layer for mangroves.
mangrove_vis_params_solid = {
    'min': 0, 'max': 1,
    'palette': ['#00000000', 'green'], # Transparent for non-mangrove (0), solid green for mangrove (1)
    'opacity': 0.8 # Explicitly set opacity for the green.
}
map_id_dict_presence = mangrove_presence_image.getMapId(mangrove_vis_params_solid)
folium.TileLayer(
    tiles=map_id_dict_presence['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Mangrove Presence (GMW)',
    show=True # Ensure it's shown by default
).add_to(m_revised_agc)


# --- Add AGC Density layer *on top of* Mangrove Presence, with transparency ---
# The AGC layer itself should be semi-transparent so the underlying green is visible where it's not colored by AGC.
# The 'white' in AGC palette will effectively be transparent against the green mangrove layer.
map_id_dict_agc_revised = agc_density_masked.getMapId(agc_vis_params_revised)
folium.TileLayer(
    tiles=map_id_dict_agc_revised['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='AGC Density (tonnes C/ha)',
    opacity=0.7 # <--- Added opacity to allow green to show through
).add_to(m_revised_agc)


# Add threshold layers (hotspots) on top
high_carbon_vis = {'min': 0, 'max': 1, 'palette': ['#00000000', 'orange']}
high_carbon_layer = agc_density_image.gt(high_carbon_threshold).selfMask()
map_id_high = high_carbon_layer.getMapId(high_carbon_vis)
folium.TileLayer(
    tiles=map_id_high['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name=f'Hotspots (>{high_carbon_threshold} tC/ha)',
    show=True
).add_to(m_revised_agc)

extreme_carbon_vis = {'min': 0, 'max': 1, 'palette': ['#00000000', 'purple']}
extreme_carbon_layer = agc_density_image.gt(extreme_carbon_threshold).selfMask()
map_id_extreme = extreme_carbon_layer.getMapId(extreme_carbon_vis)
folium.TileLayer(
    tiles=map_id_extreme['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name=f'Extreme Hotspots (>{extreme_carbon_threshold} tC/ha)',
    show=True
).add_to(m_revised_agc)


folium.LayerControl().add_to(m_revised_agc)

output_map_path_high_carbon = os.path.join(project_root, 'docs', 'high_agc_areas_map.html')
m_revised_agc.save(output_map_path_high_carbon)
print(f"Interactive high carbon areas map saved to: {output_map_path_high_carbon}")
print("Open this HTML file in your web browser to view the map.")

--- Visualizing Areas of High Aboveground Carbon (AGC) Density (Revised) ---
Interactive high carbon areas map saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\docs\high_agc_areas_map.html
Open this HTML file in your web browser to view the map.
